# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、検索と比べる

講座の用語集（`data/glossary.json`、58 語）から `scripts/make_sft_dataset.py` で Q&A データを作り、`Qwen/Qwen3.5-0.8B` を LoRA で学習します。評価用の質問（`lectures/02/advanced/eval_glossary.json`、24 問）で学習前後を比べ、最後に「用語集を検索して定義を返すだけの仕組み」と同じ質問で比べます。

| 評価の種類 | 問数 | 正解の条件 |
| --- | --- | --- |
| `seen` 用語集にある語（学習データと同じ聞き方） | 8 | 定義の重要語句（`must_include`）をすべて含む |
| `paraphrase` 言い回しや別名を変えた質問 | 8 | 同上 |
| `unknown` 用語集に無い語 | 8 | 「用語集に無い」と答える（定義を作らない） |

**進め方**: ランタイムを T4 GPU にして、上から順に実行します（全体で約 15〜20 分。学習は 5〜8 分）。`USE_DRIVE = True` にすると結果を Google Drive にも保存します。

## 0. 環境の確認とライブラリの導入

演習のノートブックと同じ版に固定します。pip の赤い依存警告（`cudf` や `pyarrow`）は無視して構いません。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao は peft 0.21 と非互換なので外す
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

- `num_train_epochs: 3`: 学習データ 116 件（58 語 × 質問 2 通り）を 3 周します。1 周 15 ステップで合計 45 ステップです。演習（14 件 × 20 周）よりデータが多いので、周回を減らして時間を揃えています。
- `add_unknown_examples`: `True` にすると、用語集に無い語の質問に「用語集に記載がありません」と答える例を学習データに足します（評価の `unknown` とは別の語）。まず `False` で 1 回実行し、余裕があれば `True` でもう 1 回実行して比べると、考察の材料になります。

In [ ]:
from pathlib import Path
import os, json, time, subprocess

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 3,
    "warmup_steps": 5,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,          # 定義文は長いので演習（96）より増やす
    "gen": {"do_sample": False},
    "add_unknown_examples": False,  # True にすると「用語集に無い」と答える例を学習データに足す
    "repo_url": "https://github.com/vhsvhafmwf/lecture-ai-engineering-2026.git",
    "repo_branch": "lec02",
    "work_dir": "/content/glossary_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
RUN_TAG = "unk" if CONFIG["add_unknown_examples"] else "base"   # 結果ファイル名に付ける（2 回実行しても上書きしない）

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. データの準備

リポジトリを取得し、`scripts/make_sft_dataset.py` で学習データ（`messages` 形式の JSONL）を作ります。評価用の質問も同じリポジトリから読み込みます。

In [ ]:
REPO = Path("/content/aie")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", CONFIG["repo_branch"], CONFIG["repo_url"], str(REPO)], check=True)
sft_text = subprocess.run(["python3", "scripts/make_sft_dataset.py"], cwd=REPO, capture_output=True, text=True, check=True).stdout
(WORK / "glossary_sft.jsonl").write_text(sft_text, encoding="utf-8")
train_records = [json.loads(line) for line in sft_text.splitlines() if line.strip()]

GLOSSARY = json.loads((REPO / "data" / "glossary.json").read_text(encoding="utf-8"))["terms"]
EVAL = json.loads((REPO / "lectures" / "02" / "advanced" / "eval_glossary.json").read_text(encoding="utf-8"))["questions"]
DEF_BY_TERM = {t["term"]: t["definition"] for t in GLOSSARY}

# 用語集に無い語の例（評価の unknown とは重ならない語）。add_unknown_examples=True のときだけ足す
UNKNOWN_ANSWER = "用語集に記載がありません。講座の資料か運営に確認してください。"
UNKNOWN_TRAIN_TERMS = ["ビームサーチ", "アテンション", "Docker", "Kubernetes", "lecNN-final タグ", "MCPプロキシ", "宿題の4レベル", "TestRunner"]
if CONFIG["add_unknown_examples"]:
    for term in UNKNOWN_TRAIN_TERMS:
        for tmpl in ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]:
            train_records.append({"messages": [{"role": "user", "content": tmpl.format(term=term)}, {"role": "assistant", "content": UNKNOWN_ANSWER}]})

train_questions = {r["messages"][0]["content"] for r in train_records}
assert all((q["question"] in train_questions) == (q["category"] == "seen") for q in EVAL), "seen だけが学習データと同じ質問になっているはず"
print(f"用語数={len(GLOSSARY)} 学習データ={len(train_records)} 件 | 評価={len(EVAL)} 問")
print("学習データの 1 件:", json.dumps(train_records[0], ensure_ascii=False)[:200], "...")

## 3. モデルの読み込みと学習前評価

学習データには system プロンプトが無いので、評価も user の質問だけで生成します（学習と評価の条件を揃えるため）。

採点は次のとおりです。
- `seen`・`paraphrase`: 定義の重要語句（`must_include`）をすべて含めば正解。`exact` は定義文と完全に一致したか（丸暗記の度合い）。
- `unknown`: 「用語集に」「記載がありません」「わかりません」などを含めば正解（定義をでっち上げなかった）。

In [ ]:
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True

def generate(model, question):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

REFUSAL_WORDS = ["用語集に", "記載がありません", "載っていません", "わかりません", "分かりません", "存在しません"]

def score(q, answer):
    if q["category"] == "unknown":
        return any(w in answer for w in REFUSAL_WORDS), False
    ok = all(w in answer for w in q["must_include"])
    return ok, answer.strip() == DEF_BY_TERM[q["term"]]

def summarize(rows, label):
    df = pd.DataFrame(rows)
    out = {"label": label}
    for cat in ["seen", "paraphrase", "unknown"]:
        out[cat] = round(df[df.category == cat].ok.mean(), 3)
    out["exact"] = int(df.exact.sum())
    return out

def evaluate(answer_fn, label):
    rows, t0 = [], time.time()
    for q in EVAL:
        ans = answer_fn(q["question"])
        ok, exact = score(q, ans)
        rows.append({"category": q["category"], "question": q["question"], "answer": ans, "ok": ok, "exact": exact})
    summary = summarize(rows, label) | {"seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}_{RUN_TAG}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary, rows

def mark(ok): return "○" if ok else "×"

before_summary, before_rows = evaluate(lambda q: generate(model, q), "before")
print(before_summary)
display(pd.DataFrame([{"種類": r["category"], "質問": r["question"], "学習前の出力": r["answer"], "判定": mark(r["ok"])} for r in before_rows]))

## 4. LoRA 学習（T4 で 5〜8 分）

演習と同じ LoRA の設定で学習します。最終 loss と学習時間は結果ファイルにも記録します。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=5, save_steps=15, save_total_limit=2,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
LOSSES = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | 最終 loss: {LOSSES[-1]:.4f}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後評価

学習前と同じ 24 問・同じ生成設定で評価し、種類ごとの正解率を並べます。

**見るところ**
- `seen` と `paraphrase` の差: 学習データと同じ聞き方なら答えられても、聞き方を変えると落ちるか。
- `unknown`: 学習データに「用語集に無い」例が無いと、知らない語にも定義らしい文を作って答えるか。それは用語集の別の語の定義を流用したものか。

In [ ]:
after_summary, after_rows = evaluate(lambda q: generate(tuned, q), "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label"))
display(pd.DataFrame([{"種類": b["category"], "質問": b["question"], "学習前": mark(b["ok"]), "学習後": mark(a["ok"]), "学習後の出力": a["answer"]}
                      for b, a in zip(before_rows, after_rows)]))

## 6. 用語集を検索する仕組みと比べる

モデルを学習させずに、質問の中に用語集の語（見出し語・括弧を外した見出し語・別名）が含まれていれば、その定義をそのまま返す仕組みを作ります。複数の語が見つかったら、いちばん長い語を選びます。見つからなければ「用語集に記載がありません」と返します。

英数字だけの語（`Pi`、`MCP` など）は前後が英数字でないときだけ一致とみなします（`API` の中の `pi` を拾わないため）。1 文字の別名（`r`）は使いません。

In [ ]:
import unicodedata

def norm(s): return unicodedata.normalize("NFKC", s).lower()

def names(t):
    ns = {t["term"], re.sub(r"（.*?）", "", t["term"]).strip(), *t["aliases"]}
    return [norm(n) for n in ns if len(n) >= 2]

def find_term(question):
    ql, best = norm(question), None
    for t in GLOSSARY:
        for n in names(t):
            hit = re.search(rf"(?<![a-z0-9]){re.escape(n)}(?![a-z0-9])", ql) if n.isascii() else n in ql
            if hit and (best is None or len(n) > best[0]):
                best = (len(n), t)
    return best[1] if best else None

def retrieve(question):
    t = find_term(question)
    return t["definition"] if t else UNKNOWN_ANSWER

search_summary, search_rows = evaluate(retrieve, "search")
display(pd.DataFrame([before_summary, after_summary, search_summary]).set_index("label"))
display(pd.DataFrame([{"種類": a["category"], "質問": a["question"], "学習後": mark(a["ok"]), "検索": mark(s["ok"]),
                       "検索が選んだ語": (find_term(s["question"]) or {}).get("term", "（なし）")} for a, s in zip(after_rows, search_rows)]))

## 7. 結果の保存

`result_advanced_<base または unk>.json` に設定・学習時間・loss・3 つの評価の要約をまとめます。質問ごとの出力は `eval_before_*.json`・`eval_after_*.json`・`eval_search_*.json` にあります（`/content/glossary_lab`。`USE_DRIVE = True` なら Drive にもコピーします）。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "train_records": len(train_records),
    "train_seconds": TRAIN_SECONDS,
    "losses": LOSSES,
    "final_loss": LOSSES[-1],
    "before": before_summary, "after": after_summary, "search": search_summary,
    "executed_training_myself": True,
}
name = f"result_advanced_{RUN_TAG}.json"
(WORK / name).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for f in [name] + [f"eval_{l}_{RUN_TAG}.json" for l in ("before", "after", "search")]:
        shutil.copy(WORK / f, Path(CONFIG["drive_dir"]) / f)
    print("Drive に保存しました:", CONFIG["drive_dir"])